# Project 4: Healthcare Analytics
**Domain:** Healthcare  
**Dataset Source:** `data/healthcare_covid.csv`  

---

## Executive Overview
COVID-19 trends and correlation analysis.

---


In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.statistical_analysis import StatisticalAnalyzer
from src.visualization import Visualizer

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')


## 1. Data Quality & Preprocessing
Checklist:
✓ Dataset shape
✓ Missing values
✓ Duplicate rows
✓ Numeric outliers


In [ ]:
loader = DataLoader('../data/healthcare_covid.csv')
df = loader.load_data()

print("==============================")
print("     RAW DATA QUALITY")
print("==============================")
print(loader.generate_data_quality_report())


In [ ]:
df = loader.clean_missing_values({})

print("==============================")
print("   CLEANED DATA QUALITY")
print("==============================")
print(loader.generate_data_quality_report())


### Outlier Analysis
**Business Question:** Are there spikes in new cases?


In [ ]:
sns.boxplot(data=df, x='NewCases')
plt.show()

**Finding:** Massive spikes (outliers) exist.  
**Meaning:** These represent distinct 'waves' of the pandemic.  
**Recommendation:** Ensure surge capacity during wave events.

### Advanced Pandas: Melt
**Business Question:** How do cases, recoveries, and deaths compare in a normalized format?


In [ ]:
melted = df.melt(id_vars=['Date', 'State'], value_vars=['NewCases', 'Recoveries', 'Deaths'])
display(melted.head())

### Trend Analysis
**Business Question:** How are cases trending over time?


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df.groupby('Date')['NewCases'].sum().plot()
plt.title('Total Cases Over Time')
plt.show()

### Statistical Hypothesis Testing
**Business Question:** Is vaccination associated with lower positivity?


In [ ]:
stats = StatisticalAnalyzer(df)
res = stats.pearson_correlation_test('VaccinationDosesAdministered', 'PositivityRate_Pct')
print(stats.format_hypothesis_report(
    'No correlation between vax and positivity.', 'Negative correlation exists.', 'Pearson Correlation', 'r', res['r_statistic'], res['p_value'], 'Higher vaccination rates are associated with lower positivity rates.', 'No statistically significant linear relationship was detected between vaccination doses and positivity rate.', why_it_matters_reject='Suggests the efficacy of vaccination campaigns on community spread.', ci_lower=res['ci_lower'], ci_upper=res['ci_upper']
))

## Limitations
- Dataset size is limited.
- Results are observational.
- Correlation does not imply causation.
- Some variables contain missing observations.
- External factors are not included.
